# CEPHA29-MD — masked multi-domain fine-tune (all 29 landmarks)

Fine-tunes the 29-landmark model on **Aariz (all 29)** + **ISBI 2015 (only the 19
shared)**, using a per-sample visibility mask so the 10 Aariz-only landmarks are
not supervised on ISBI images. This keeps full 29-landmark production parity while
gaining the domain diversity that made E1 (the 19-landmark model) a clear win.

- Vocabulary: CEPHA29 (unchanged channel order).
- Dataset: `Merged29` annotations; ISBI JSONs carry `visible: [19 symbols]`.
- Patches: `scripts/ai/cephalohrnet_mask_patch.py` (dataset mask, masked loss,
  masked metrics).
- Starts from the original 29-landmark `best.pt` (head loaded too — shapes match).
- Held-out evaluation: checkpoints are chosen on Aariz `valid` + 40 ISBI `train`
  images; scores are reported on Aariz `test` and ISBI `test`, which training and
  checkpoint selection never see.
- The current app model (md19) and the original 29-landmark model are fetched from
  Hugging Face and scored on the same test sets, so one table decides the switch.

**Kaggle setup**: GPU T4 x2, Internet On, attach `best.pt` (or `best.zip`).
ISBI/Aariz are downloaded automatically.

Outputs: `/kaggle/working/deliverables_md29/`.

In [ ]:
!pip -q install timm albumentations opencv-python-headless pyyaml onnx onnxruntime pillow numpy scipy pandas requests seaborn
print("deps installed")

In [ ]:
import os
%cd /kaggle/working
# The mask patch edits CephaloHRNet source by exact match, so pin the commit it was
# validated against (an upstream change could otherwise break the patch mid-run).
CEPHALOHRNET_SHA = "a852d46b43c80796254588d3a8c8df57d30fe1ad"
!rm -rf CephaloHRNet CephalometryStudio
!git init -q CephaloHRNet
!git -C CephaloHRNet fetch -q --depth 1 https://github.com/Cestovatels/CephaloHRNet.git {CEPHALOHRNET_SHA}
!git -C CephaloHRNet checkout -q FETCH_HEAD
!git clone --depth 1 https://github.com/Muhammad-Nabeel-Sh/CephalometryStudio.git
assert os.path.exists("CephaloHRNet/models/hrnet.py")
import subprocess
head = subprocess.run(["git", "-C", "CephaloHRNet", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == CEPHALOHRNET_SHA, head
assert os.path.exists("CephalometryStudio/scripts/ai/cephalohrnet_mask_patch.py"), "push the mask patch first"
print("cloned")

In [ ]:
import glob, os, sys, urllib.request
import numpy as np
sys.path.insert(0, "CephalometryStudio/scripts/ai")
from canonical import CEPHA29, ISBI19, ISBI_TO_CEPHA

BASE = "https://raw.githubusercontent.com/stolariks/medical-landmark-detection/main/data/isbi-2015/"
ANN_TRAIN = "/kaggle/working/isbi_train.csv"
ANN_TEST = "/kaggle/working/isbi_test.csv"
for url, dest in [(BASE + "train/annotations.csv", ANN_TRAIN),
                  (BASE + "test/annotations.csv", ANN_TEST)]:
    if not os.path.exists(dest):
        req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
        with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
            f.write(r.read())

def read_isbi_csv(path):
    import csv
    out = {}
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.reader(f):
            if not row or row[0] == "image_path":
                continue
            try:
                pts = np.array([float(v) for v in row[1:]], np.float64).reshape(-1, 2)
            except Exception:
                continue
            if pts.shape[0] == len(ISBI19):
                out[os.path.splitext(os.path.basename(row[0]))[0]] = pts
    return out

isbi_gt = {"train": read_isbi_csv(ANN_TRAIN), "test": read_isbi_csv(ANN_TEST)}
print("ISBI GT:", {k: len(v) for k, v in isbi_gt.items()})

def find_input_images(stems):
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for p in glob.glob("/kaggle/input/**/" + ext, recursive=True):
            if os.path.splitext(os.path.basename(p))[0] in stems:
                return os.path.dirname(p)
    return None

ISBI_DIR = find_input_images(set(isbi_gt["train"]) | set(isbi_gt["test"]))
print("ISBI image dir:", ISBI_DIR or "(fetch from mirror)")

def isbi_image(split, stem):
    if ISBI_DIR:
        for ext in (".jpg", ".jpeg", ".png"):
            p = os.path.join(ISBI_DIR, stem + ext)
            if os.path.exists(p):
                return p
        return None
    cache = "/kaggle/working/isbi_img_" + split
    os.makedirs(cache, exist_ok=True)
    for ext in (".jpg", ".png"):
        dest = os.path.join(cache, stem + ext)
        if os.path.exists(dest):
            return dest
        try:
            url = BASE + split + "/cepha400/" + stem + ext
            req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
                f.write(r.read())
            return dest
        except Exception:
            continue
    return None

isbi_cases = {"train": [], "test": []}
for split in ("train", "test"):
    for stem, gt in isbi_gt[split].items():
        p = isbi_image(split, stem)
        if p:
            isbi_cases[split].append((stem, p, gt))
    print("ISBI", split, "cases:", len(isbi_cases[split]))

def find_dataset(root):
    for base, dirs, _ in os.walk(root):
        for split in ("train", "valid"):
            if os.path.isdir(os.path.join(base, split, "Cephalograms")):
                return base
    return None

AARIZ = find_dataset("/kaggle/input")
if AARIZ is None:
    import zipfile
    URL = "https://ndownloader.figshare.com/files/51041642"
    ZIP = "/kaggle/working/Aariz.zip"
    if not os.path.exists(ZIP) or os.path.getsize(ZIP) != 2098209792:
        req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=900) as r, open(ZIP, "wb") as f:
            while True:
                c = r.read(1 << 20)
                if not c:
                    break
                f.write(c)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/kaggle/working/aariz")
    AARIZ = find_dataset("/kaggle/working/aariz")
print("AARIZ:", AARIZ)
assert AARIZ, "Aariz dataset not found"

In [ ]:
import glob, json, os, shutil
import numpy as np

MD = "/kaggle/working/md29"
EVAL = "/kaggle/working/eval29"
for d in (MD, EVAL):
    shutil.rmtree(d, ignore_errors=True)

def ann_dir(root):
    # CephaloHRNet resolves --annotators Merged29 under this directory.
    return os.path.join(root, "Annotations", "Cephalometric Landmarks", "Merged29")

def read_syms(path):
    data = json.load(open(path, encoding="utf-8"))
    return {lm["symbol"]: (float(lm["value"]["x"]), float(lm["value"]["y"]))
            for lm in data.get("landmarks", []) if "x" in (lm.get("value") or {})}

def aariz_avg(split, stem):
    root = os.path.join(AARIZ, split, "Annotations", "Cephalometric Landmarks")
    dirs = [d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(d)]
    js = [os.path.join(d, stem + ".json") for d in dirs
          if os.path.exists(os.path.join(d, stem + ".json"))]
    if not js:
        return None
    sets = [read_syms(p) for p in js]
    keys = set(sets[0])
    for s in sets[1:]:
        keys &= set(s)
    return {k: (float(np.mean([s[k][0] for s in sets])),
                float(np.mean([s[k][1] for s in sets]))) for k in keys}

def write_case(root, prefix, domain, stem, img_path, pts29, visible=None):
    ceph = os.path.join(root, "Cephalograms")
    ad = ann_dir(root)
    os.makedirs(ceph, exist_ok=True)
    os.makedirs(ad, exist_ok=True)
    new_stem = (domain + "_" + stem) if prefix else stem
    ext = os.path.splitext(img_path)[1].lower()
    if ext not in (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"):
        ext = ".jpg"
    dst = os.path.join(ceph, new_stem + ext)
    if not os.path.exists(dst):
        shutil.copy(img_path, dst)
    doc = {"ceph_id": new_stem,
           "landmarks": [{"symbol": s, "value": {"x": pts29[s][0], "y": pts29[s][1]}} for s in CEPHA29]}
    if visible is not None:
        doc["visible"] = visible
    json.dump(doc, open(os.path.join(ad, new_stem + ".json"), "w"))

def aariz_pts(split, stem):
    avg = aariz_avg(split, stem)
    if not avg or not all(s in avg for s in CEPHA29):
        return None
    return {s: avg[s] for s in CEPHA29}

def write_eval19(root, stem, img_path, pts29):
    # Aariz case for 19-channel models: the 19 shared landmarks under ISBI symbols.
    ceph = os.path.join(root, "Cephalograms")
    ad = os.path.join(root, "Annotations", "Cephalometric Landmarks", "Merged19")
    os.makedirs(ceph, exist_ok=True)
    os.makedirs(ad, exist_ok=True)
    shutil.copy(img_path, os.path.join(ceph, stem + os.path.splitext(img_path)[1].lower()))
    doc = {"ceph_id": stem,
           "landmarks": [{"symbol": s, "value": {"x": pts29[ISBI_TO_CEPHA[s]][0], "y": pts29[ISBI_TO_CEPHA[s]][1]}}
                         for s in ISBI19]}
    json.dump(doc, open(os.path.join(ad, stem + ".json"), "w"))

def isbi_pts(gt):
    # Map the 19 ISBI landmarks onto CEPHA29; the other 10 are masked sentinels.
    mapped = {ISBI_TO_CEPHA[s]: (float(gt[i][0]), float(gt[i][1])) for i, s in enumerate(ISBI19)}
    pts = {s: mapped.get(s, (-1.0, -1.0)) for s in CEPHA29}
    visible = [s for s in CEPHA29 if s in mapped]
    return pts, visible

# Splits — the test sets stay out of training AND checkpoint selection:
#   train: Aariz train + ISBI train (minus 40 held out)
#   valid: Aariz valid + the 40 held-out ISBI train images (chooses best.pt)
#   eval:  Aariz test (29) + ISBI test (19 shared) — reported only.
assert os.path.isdir(os.path.join(AARIZ, "test", "Cephalograms")), "Aariz test split not found"
ISBI_VAL = 40
isbi_train = sorted(isbi_cases["train"], key=lambda c: c[0])
hold = set(np.random.default_rng(29).choice(len(isbi_train), size=ISBI_VAL, replace=False).tolist())
isbi_split = {"train": [c for i, c in enumerate(isbi_train) if i not in hold],
              "valid": [c for i, c in enumerate(isbi_train) if i in hold]}

for split in ("train", "valid"):
    for p in sorted(glob.glob(os.path.join(AARIZ, split, "Cephalograms", "*"))):
        stem = os.path.splitext(os.path.basename(p))[0]
        pts = aariz_pts(split, stem)
        if pts:
            write_case(os.path.join(MD, split), True, "aariz", stem, p, pts)
    for stem, p, gt in isbi_split[split]:
        pts, visible = isbi_pts(gt)
        write_case(os.path.join(MD, split), True, "isbi", stem, p, pts, visible)

for p in sorted(glob.glob(os.path.join(AARIZ, "test", "Cephalograms", "*"))):
    stem = os.path.splitext(os.path.basename(p))[0]
    pts = aariz_pts("test", stem)
    if pts:
        write_case(os.path.join(EVAL, "aariz29"), False, "aariz", stem, p, pts)
        write_eval19(os.path.join(EVAL, "aariz19"), stem, p, pts)
for stem, p, gt in isbi_cases["test"]:
    pts, visible = isbi_pts(gt)
    write_case(os.path.join(EVAL, "isbi19"), False, "isbi", stem, p, pts, visible)
if os.path.isdir(os.path.join(EVAL, "isbi19")):
    shutil.copy(ANN_TEST, os.path.join(EVAL, "isbi19", "annotations.csv"))

def count(root):
    return len(glob.glob(os.path.join(root, "Cephalograms", "*")))

def stems(root, strip_prefix):
    out = set()
    for p in glob.glob(os.path.join(root, "Cephalograms", "*")):
        s = os.path.splitext(os.path.basename(p))[0]
        out.add(s.split("_", 1)[1] if strip_prefix else s)
    return out

print("md29 train:", count(os.path.join(MD, "train")),
      "valid:", count(os.path.join(MD, "valid")),
      f"(ISBI {len(isbi_split['train'])}/{len(isbi_split['valid'])})",
      "| eval aariz29:", count(os.path.join(EVAL, "aariz29")),
      "aariz19:", count(os.path.join(EVAL, "aariz19")),
      "isbi19:", count(os.path.join(EVAL, "isbi19")))
assert count(os.path.join(MD, "train")) > 0 and count(os.path.join(MD, "valid")) > 0
assert count(os.path.join(EVAL, "aariz29")) > 0 and count(os.path.join(EVAL, "isbi19")) > 0
# No image may be both trained/selected on and reported.
seen = stems(os.path.join(MD, "train"), True) | stems(os.path.join(MD, "valid"), True)
reported = stems(os.path.join(EVAL, "aariz29"), False) | stems(os.path.join(EVAL, "isbi19"), False)
assert not (seen & reported), sorted(seen & reported)[:5]
print("no overlap between train/valid and the test sets")

In [ ]:
import sys
sys.path.insert(0, "CephalometryStudio/scripts/ai")
from cephalohrnet_mask_patch import apply_mask_patches
apply_mask_patches("/kaggle/working/CephaloHRNet")
print("mask patches applied")

In [ ]:
import glob, os, shutil
import torch

pref = {"best.pt": 0, "best.pth": 1, "best.zip": 2}
cands = []
for name, rank in pref.items():
    for p in glob.glob("/kaggle/input/**/" + name, recursive=True):
        cands.append((rank, p))
cands.sort()
BEST = cands[0][1] if cands else None
if BEST and BEST.lower().endswith(".zip"):
    dest = "/kaggle/working/best.pt"
    if not os.path.exists(dest):
        shutil.copy(BEST, dest)
    BEST = dest
print("best.pt:", BEST)
assert BEST, "Attach the 29-landmark best.pt (or best.zip) as a Kaggle dataset"
ck = torch.load(BEST, map_location="cpu", weights_only=False)
n_head = ck["model"]["head.weight"].shape[0]
print("checkpoint head channels:", n_head, "| epoch:", ck.get("epoch"))
assert n_head == 29, "expected the 29-landmark checkpoint"

In [ ]:
import glob, os, shutil
import numpy as np
# Static INT8 calibration from BOTH domains: sorted order alone would pick only
# aariz_* files, tuning the quantization ranges to one scanner.
calib = "/kaggle/working/calib_md29"
shutil.rmtree(calib, ignore_errors=True)
os.makedirs(calib, exist_ok=True)
rng = np.random.default_rng(7)
for domain in ("aariz", "isbi"):
    files = sorted(glob.glob(os.path.join(MD, "train", "Cephalograms", domain + "_*")))
    for i in sorted(rng.choice(len(files), size=min(16, len(files)), replace=False).tolist()):
        shutil.copy(files[i], os.path.join(calib, os.path.basename(files[i])))
print("calib:", {d: len(glob.glob(os.path.join(calib, d + "_*"))) for d in ("aariz", "isbi")})

In [ ]:
import glob, os, subprocess, sys

for split in ("train", "valid"):
    n_img = len(glob.glob(os.path.join(MD, split, "Cephalograms", "*")))
    n_ann = len(glob.glob(os.path.join(MD, split, "Annotations", "Cephalometric Landmarks", "Merged29", "*.json")))
    print(f"{split}: images={n_img} annotations={n_ann}")
    assert n_img and n_ann, f"{split} split is empty"

cmd = [sys.executable, "train.py",
       "--data", MD, "--model", "hrnet_w32",
       "--imgsz", "768", "--batch", "4", "--epochs", "80",
       "--lr", "5e-4",
       "--loss", "awing", "--amp", "--device", "0",
       "--workers", "0", "--patience", "15",
       "--project", "runs/train", "--name", "md29_w32",
       "--annotators", "Merged29", "--img-dir", "Cephalograms",
       "--pretrained-path", BEST]
print(" ".join(cmd))
r = subprocess.run(cmd, cwd="/kaggle/working/CephaloHRNet", capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-6000:])
r.check_returncode()

In [ ]:
import glob, os, subprocess, sys
BESTR = sorted(glob.glob("/kaggle/working/CephaloHRNet/runs/train/md29_*/best.pt"),
               key=os.path.getmtime)[-1]
SAMPLE = sorted(glob.glob(os.path.join(MD, "train", "Cephalograms", "*")))[0]
print("weights:", BESTR)
subprocess.run([sys.executable, "CephalometryStudio/scripts/export-landmark-onnx.py",
                "--repo", "/kaggle/working/CephaloHRNet",
                "--weights", BESTR, "--model", "hrnet_w32",
                "--num-landmarks", "29", "--input-size", "768",
                "--out", "/kaggle/working/landmarks-md29.onnx",
                "--quantize", "--calib-dir", "/kaggle/working/calib_md29",
                "--verify", SAMPLE],
               cwd="/kaggle/working", check=True)

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "CephalometryStudio/scripts/build-shape-model.py",
                "--set", "cepha29",
                "--ann-dir", os.path.join(AARIZ, "train", "Annotations", "Cephalometric Landmarks", "Senior Orthodontists"),
                "--k", "10",
                "--out", "/kaggle/working/shapeModel.cepha29.json"],
               cwd="/kaggle/working", check=True)

In [ ]:
import hashlib, json, os, subprocess, sys, urllib.request
OUT = "/kaggle/working/audit_md29"
os.makedirs(OUT, exist_ok=True)

# Reference models from the app's Hugging Face repo (SHA-256 as in landmarkModelInfo.js).
HF = "https://huggingface.co/MuhammadNabeelSh/cephalometry-landmarks/resolve/main/"
REFS = {
    "landmarks-md19.int8.onnx": "aabfddcebdb0aad0f14bc99bba7498fd2a7f9a57135d86180820c91c57ed7315",
    "landmarks-cepha29.int8.onnx": "f7f1db9cc2de37c0fa94dab431d7d3446d1d8ffc62fa92e6610c2380af83080d",
}
for name, digest in REFS.items():
    dest = "/kaggle/working/ref_" + name
    if not os.path.exists(dest):
        urllib.request.urlretrieve(HF + name, dest)
    got = hashlib.sha256(open(dest, "rb").read()).hexdigest()
    assert got == digest, (name, got)

MODELS = [  # (tag, onnx, vocabulary)
    ("md29_int8", "/kaggle/working/landmarks-md29.int8.onnx", "cepha29"),
    ("md29_fp32", "/kaggle/working/landmarks-md29.onnx", "cepha29"),
    ("md19_int8 (E1, app now)", "/kaggle/working/ref_landmarks-md19.int8.onnx", "cepha19"),
    ("cepha29_int8 (original)", "/kaggle/working/ref_landmarks-cepha29.int8.onnx", "cepha29"),
]
ERR = []
for tag, model, vocab in MODELS:
    key = tag.split(" ")[0]
    os.makedirs(os.path.join(OUT, key), exist_ok=True)
    # External: ISBI test (19 shared), raw, production search.
    r = subprocess.run([sys.executable, "CephalometryStudio/scripts/ai/external_benchmark.py",
                        "--model", model,
                        "--isbi-images", os.path.join(EVAL, "isbi19", "Cephalograms"),
                        "--isbi-ann", os.path.join(EVAL, "isbi19", "annotations.csv"),
                        "--search", "--no-ssm",
                        "--out", os.path.join(OUT, key, "isbi19")],
                       cwd="/kaggle/working")
    if r.returncode: ERR.append(key + " isbi")
    # In-domain: Aariz test, raw (29 landmarks, or the shared 19 for md19).
    sub = "aariz29" if vocab == "cepha29" else "aariz19"
    ann = "Merged29" if vocab == "cepha29" else "Merged19"
    r = subprocess.run([sys.executable, "CephalometryStudio/scripts/eval-landmark-model.py",
                        "--model", model,
                        "--set", vocab, "--limit", "200", "--dark",
                        "--ann-dir", os.path.join(EVAL, sub, "Annotations", "Cephalometric Landmarks", ann),
                        "--images", os.path.join(EVAL, sub, "Cephalograms"),
                        "--metrics-out", os.path.join(OUT, key, "aariz_metrics.json")],
                       cwd="/kaggle/working")
    if r.returncode: ERR.append(key + " aariz")
print("failed:", ERR)

In [ ]:
import json, os
import numpy as np
import pandas as pd

SHARED = [ISBI_TO_CEPHA[s] for s in ISBI19]           # CEPHA29 symbols of the shared 19
EXTRA = [s for s in CEPHA29 if s not in SHARED]       # the 10 Aariz-only landmarks
TO_ISBI = {c: s for s, c in ISBI_TO_CEPHA.items()}

def load(path):
    try:
        return json.load(open(path))
    except Exception:
        return None

def lm_mre(per, syms, rename=None):
    vals = [per.get((rename or {}).get(s, s), {}).get("mre_mm") for s in syms]
    vals = [v for v in vals if isinstance(v, (int, float)) and np.isfinite(v)]
    return round(float(np.mean(vals)), 2) if len(vals) == len(syms) else None

rows = []
for tag, _, vocab in MODELS:
    key = tag.split(" ")[0]
    isbi = load(os.path.join(OUT, key, "isbi19", "isbi_metrics.json")) or {}
    aar = load(os.path.join(OUT, key, "aariz_metrics.json")) or {}
    io, ao, per = isbi.get("overall", {}), aar.get("overall", {}), aar.get("per_landmark", {})
    rename = TO_ISBI if vocab == "cepha19" else None
    rows.append({
        "model": tag,
        "ISBI MRE": io.get("mre_mm"), "ISBI median": io.get("median_mm"), "ISBI SDR@2": io.get("sdr_2.0mm"),
        "Aariz shared-19 MRE": lm_mre(per, SHARED, rename),
        "Aariz extra-10 MRE": lm_mre(per, EXTRA) if vocab == "cepha29" else None,
        "Aariz all MRE": ao.get("mre_mm"), "Aariz median": ao.get("median_mm"), "Aariz SDR@2": ao.get("sdr_2.0mm"),
    })
table = pd.DataFrame(rows).set_index("model").astype(float).round(2)
table.to_csv(os.path.join(OUT, "comparison.csv"))
print("Held-out test sets: ISBI test (19 shared) and Aariz test.")
print("'Aariz all' is 29 landmarks for 29-channel models and the shared 19 for md19.")
table

In [ ]:
import glob, hashlib, os, shutil
out = "/kaggle/working/deliverables_md29"
os.makedirs(out, exist_ok=True)
for name in ("landmarks-md29.onnx", "landmarks-md29.int8.onnx", "landmarks-md29-verify.png",
             "shapeModel.cepha29.json"):
    for p in glob.glob("/kaggle/working/" + name):
        shutil.copy(p, out)
# The trained checkpoint: the starting point for any later fine-tune (e.g. more landmarks).
shutil.copy(BESTR, os.path.join(out, "best_md29.pt"))
for p in glob.glob("/kaggle/working/audit_md29/**/*", recursive=True):
    if os.path.isfile(p):
        rel = os.path.relpath(p, "/kaggle/working/audit_md29")
        dst = os.path.join(out, "audit", rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(p, dst)

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

print("=== DELIVERABLES ===")
for f in sorted(os.listdir(out)):
    p = os.path.join(out, f)
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:9.2f} MB  {f}\n            sha256 {sha(p)}")

## Next

1. Download `/kaggle/working/deliverables_md29/` (includes `best_md29.pt` — keep it).
2. Read `audit/comparison.csv` (the table above). All rows are scored on the same
   held-out images: ISBI test (19 shared) and Aariz test.
   - **ISBI** — md29 should match or beat md19 (E1).
   - **Aariz shared-19** — md29 should be no worse than md19.
   - **Aariz extra-10** — md29 should be no worse than the original cepha29 model.
   - **md29_fp32 vs md29_int8** — the quantization cost.
3. If it passes, it becomes the production model: host `landmarks-md29.int8.onnx` and
   switch `LANDMARK_MODEL` back to a 29-set config.
4. Inspect `landmarks-md29-verify.png` (29 channels, CEPHA29 order).